# 1. Data Loading

## 1.1. Fetching Weather Data

In [ ]:
import pandas as pd
import requests

cities = [
    {"name": "Cairo", "lat": 30.0444, "lon": 31.2357, "id": 0},
    {"name": "Alexandria", "lat": 31.2001, "lon": 29.9187, "id": 1},
    {"name": "Luxor", "lat": 25.6872, "lon": 32.6396, "id": 2},
    {"name": "Aswan", "lat": 24.0889, "lon": 32.8998, "id": 3},
    {"name": "Ismailia", "lat": 30.5965, "lon": 32.2715, "id": 4},
    {"name": "Sharm", "lat": 27.9158, "lon": 34.3300, "id": 5},
]

url = "https://archive-api.open-meteo.com/v1/archive"
dfs = []

for city in cities:
    print(f"Fetching data for {city['name']}...")
    params = {
        "latitude": city["lat"],
        "longitude": city["lon"],
        "hourly": ["temperature_2m", "weathercode"],
        "past_days": 730,
        "forecast_days": 1
    }
    response = requests.get(url, params=params)
    response.raise_for_status()
    data = response.json()

    df = pd.DataFrame(data['hourly'])
    df['time'] = pd.to_datetime(df['time'])
    df['city_id'] = city['id']
    df['city_name'] = city['name']

    dfs.append(df)

Fetching data for Cairo...
Fetching data for Alexandria...
Fetching data for Luxor...
Fetching data for Aswan...
Fetching data for Ismailia...
Fetching data for Sharm...


## 1.2. Data Exploration

In [2]:
combined_df = pd.concat(dfs, ignore_index=True)
combined_df.shape

(105264, 5)

In [3]:
combined_df.head(30000)

,time,temperature_2m,weathercode,city_id,city_name
0,2024-05-03 00:00:00,19.3,0,0,Cairo
1,2024-05-03 01:00:00,21.5,0,0,Cairo
2,2024-05-03 02:00:00,20.6,0,0,Cairo
3,2024-05-03 03:00:00,19.3,0,0,Cairo
4,2024-05-03 04:00:00,20.0,0,0,Cairo
...,...,...,...,...,...
29995,2025-10-03 19:00:00,24.1,0,1,Alexandria
29996,2025-10-03 20:00:00,23.1,0,1,Alexandria
29997,2025-10-03 21:00:00,22.1,0,1,Alexandria
29998,2025-10-03 22:00:00,21.5,0,1,Alexandria


In [4]:
combined_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 105264 entries, 0 to 105263
Data columns (total 5 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   time            105264 non-null  datetime64[ns]
 1   temperature_2m  105264 non-null  float64       
 2   weathercode     105264 non-null  int64         
 3   city_id         105264 non-null  int64         
 4   city_name       105264 non-null  object        
dtypes: datetime64[ns](1), float64(1), int64(2), object(1)
memory usage: 4.0+ MB


In [5]:
combined_df['city_name'].value_counts()

,count
city_name,
Cairo,17544
Alexandria,17544
Luxor,17544
Aswan,17544
Ismailia,17544
Sharm,17544


In [6]:
print(combined_df.duplicated().sum())

0


# 2. Preprocessing

## 2.1. Mapping Weather Codes

In [7]:
def map_weather(code):

    # Class 0: Clear sky ☀️
    if code == 0:
        return 0

    # Class 1: Overcast ⛅
    elif code == 3:
        return 1

    # Class 2: Mainly clear ⛅
    elif code == 1:
        return 2

    # Class 3: Partly cloudy ⛅
    elif code == 2:
        return 3

    # Class 4: Fog 🌫️
    elif code in [45, 48]:
        return 4

    # Class 5: Rain_Event
    else:
        return 5

In [8]:
combined_df['weathercode'] = combined_df['weathercode'].apply(map_weather)
combined_df['weathercode'].value_counts()

,count
weathercode,
0,81893
1,11362
2,7369
3,3691
5,949


## 2.2. Time Cyclic Encoding

In [9]:
combined_df.columns

Index(['time', 'temperature_2m', 'weathercode', 'city_id', 'city_name'], dtype='object')

In [10]:
print("Full Date:", combined_df['time'][1])
print("Hour:", combined_df['time'][1].hour)
print("Day of the Year:", combined_df['time'][1].dayofyear)

Full Date: 2024-05-03 01:00:00
Hour: 1
Day of the Year: 124


In [11]:
import numpy as np

def sine_encoding(value, max_value):
    return np.sin(2 * np.pi * value / max_value)

def cos_encoding(value, max_value):
    return np.cos(2 * np.pi * value / max_value)


In [12]:
# Hour Encoding
combined_df["hour_sin"] = sine_encoding(combined_df["time"].dt.hour, 24)
combined_df["hour_cos"] = cos_encoding(combined_df["time"].dt.hour, 24)

# Day of Year Encoding
combined_df["dayofyear_sin"] = sine_encoding(combined_df["time"].dt.dayofyear, 365)
combined_df["dayofyear_cos"] = cos_encoding(combined_df["time"].dt.dayofyear, 365)

print("Cyclical encoding complete!")
combined_df.head()

Cyclical encoding complete!


,time,temperature_2m,weathercode,city_id,city_name,hour_sin,hour_cos,dayofyear_sin,dayofyear_cos
0,2024-05-03 00:00:00,19.3,0,0,Cairo,0.000000,1.000000,0.845249,-0.534373
1,2024-05-03 01:00:00,21.5,0,0,Cairo,0.258819,0.965926,0.845249,-0.534373
2,2024-05-03 02:00:00,20.6,0,0,Cairo,0.500000,0.866025,0.845249,-0.534373
3,2024-05-03 03:00:00,19.3,0,0,Cairo,0.707107,0.707107,0.845249,-0.534373
4,2024-05-03 04:00:00,20.0,0,0,Cairo,0.866025,0.500000,0.845249,-0.534373


## 2.3. Data Splitting

In [ ]:
def time_split(df, train_frac=0.8, val_frac=0.9):
    train_list, val_list, test_list = [], [], []

    for _, group in df.groupby("city_id", sort=True):       # 6 Cities (80% from each city)
        group = group.sort_values("time").reset_index(drop=True)
        n = len(group)  
        train_end = int(n * train_frac)
        valid_end = int(n * val_frac)
        train_list.append(group.iloc[:train_end])
        val_list.append(group.iloc[train_end:valid_end])
        test_list.append(group.iloc[valid_end:])

    # Concatenate results
    train_df = pd.concat(train_list, ignore_index=True)
    val_df   = pd.concat(val_list,  ignore_index=True)
    test_df  = pd.concat(test_list, ignore_index=True)

    # Drop time column from each dataframe
    train_df = train_df.drop(columns=["time"])
    val_df   = val_df.drop(columns=["time"])
    test_df  = test_df.drop(columns=["time"])

    return train_df, val_df, test_df                

train_df, val_df, test_df = time_split(combined_df)

In [14]:
train_df.head()

,temperature_2m,weathercode,city_id,city_name,hour_sin,hour_cos,dayofyear_sin,dayofyear_cos
0,19.3,0,0,Cairo,0.000000,1.000000,0.845249,-0.534373
1,21.5,0,0,Cairo,0.258819,0.965926,0.845249,-0.534373
2,20.6,0,0,Cairo,0.500000,0.866025,0.845249,-0.534373
3,19.3,0,0,Cairo,0.707107,0.707107,0.845249,-0.534373
4,20.0,0,0,Cairo,0.866025,0.500000,0.845249,-0.534373


## 2.4. Scaling

In [ ]:
from sklearn.preprocessing import MinMaxScaler 

scaler = MinMaxScaler()  

# Fit ONLY on the training temperature and transform all sets
train_df["temperature_2m"] = scaler.fit_transform(train_df[["temperature_2m"]])
val_df["temperature_2m"]   = scaler.transform(val_df[["temperature_2m"]])
test_df["temperature_2m"]  = scaler.transform(test_df[["temperature_2m"]])


In [16]:
train_df.head()

,temperature_2m,weathercode,city_id,city_name,hour_sin,hour_cos,dayofyear_sin,dayofyear_cos
0,0.313953,0,0,Cairo,0.000000,1.000000,0.845249,-0.534373
1,0.365116,0,0,Cairo,0.258819,0.965926,0.845249,-0.534373
2,0.344186,0,0,Cairo,0.500000,0.866025,0.845249,-0.534373
3,0.313953,0,0,Cairo,0.707107,0.707107,0.845249,-0.534373
4,0.330233,0,0,Cairo,0.866025,0.500000,0.845249,-0.534373


## 2.5. Building Sequences

In [17]:
import numpy as np

SEQ_LEN = 24  # Look back 24 hours

def make_sequences(df, seq_len=SEQ_LEN, target_col='temperature_2m'):
    X_num_list, X_wcode_list, X_city_list, y_list = [], [], [], []

    # Standard numerical features from cyclic encoding
    num_cols = ['hour_sin', 'hour_cos', 'dayofyear_sin', 'dayofyear_cos']

    # We iterate per city to ensure sequences dont cross city boundaries
    for city_id in sorted(df['city_id'].unique()):
        city_df = df[df['city_id'] == city_id].copy()

        num_feats = city_df[num_cols].values
        wcodes = city_df['weathercode'].values
        cities = city_df['city_id'].values
        targets = city_df[target_col].values

        for i in range(seq_len, len(city_df)):               # i: 24 -> 17k
            X_num_list.append(num_feats[i - seq_len:i])      # 0->24, 1->25, 2->26, ...
            X_wcode_list.append(wcodes[i - seq_len:i])
            X_city_list.append(cities[i - seq_len:i])
            y_list.append(targets[i])

    return (
        np.array(X_num_list),   # (N, 24, 4)
        np.array(X_wcode_list), # (N, 24)
        np.array(X_city_list),  # (N, 24)
        np.array(y_list)        # (N,)
    )


In [18]:
# Apply to all three datasets
X_num_train, X_wcode_train, X_city_train, y_train_final = make_sequences(train_df)
X_num_val, X_wcode_val, X_city_val, y_val_final = make_sequences(val_df)
X_num_test, X_wcode_test, X_city_test, y_test_final = make_sequences(test_df)

In [19]:
print(f'\nFinal Train Shape: {X_num_train.shape}')
print(f'Final Val Shape:   {X_num_val.shape}')
print(f'Final Test Shape:  {X_num_test.shape}')


Final Train Shape: (84066, 24, 4)
Final Val Shape:   (10380, 24, 4)
Final Test Shape:  (10386, 24, 4)


# 3. Model

## 3.1. Model Definition

In [2]:
from tensorflow.keras.models import Model
from tensorflow.keras.regularizers import L2
from tensorflow.keras.layers import LSTM, Embedding, Input, Dense, Concatenate


SEQ_LEN = 24
# Input Time Columns (Already Encoded)
time_input = Input(shape=(SEQ_LEN, 4), name="Time Columns")

# Input and Embedding City ID Column
city_input = Input(shape=(SEQ_LEN,), name="Cities")
city_embedding = Embedding(6, 4)(city_input)

# Input and Embedding Weather Code Column
weather_input = Input(shape=(SEQ_LEN,), name="Weather Code")
weather_embedding = Embedding(6, 8)(weather_input)

# Concatenation
concatenated = Concatenate(axis=-1)([time_input, city_embedding, weather_embedding])

# LSTM Layers
x = LSTM(64, dropout=0.2, recurrent_dropout=0.2, return_sequences=True)(concatenated)
x = LSTM(32, dropout=0.2, recurrent_dropout=0.2, return_sequences=False)(x)

# Dense Layers
x = Dense(16, activation='relu', kernel_regularizer=L2(1e-4))(x)
output = Dense(1)(x)

model = Model(inputs=[time_input, city_input, weather_input], outputs=output)
model.compile(optimizer="adam", loss="mse", metrics=["mae"])
model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ Cities (InputLayer) │ (None, 24)        │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Weather Code        │ (None, 24)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Time Columns        │ (None, 24, 4)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_2         │ (None, 24, 4)     │         24 │ Cities[0][0]      │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_3         │ (None, 24, 8)     │         48 │ Weather           │
│ (Embedding)         │                   │            │ Code[0][0]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_1       │ (None, 24, 16)    │          0 │ Time              │
│ (Concatenate)       │                   │            │ Columns[0][0],    │
│                     │                   │            │ embedding_2[0][0… │
│                     │                   │            │ embedding_3[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_2 (LSTM)       │ (None, 24, 64)    │     20,736 │ concatenate_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_3 (LSTM)       │ (None, 32)        │     12,416 │ lstm_2[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 16)        │        528 │ lstm_3[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 1)         │         17 │ dense_2[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 33,769 (131.91 KB)

 Trainable params: 33,769 (131.91 KB)

 Non-trainable params: 0 (0.00 B)

## 3.2. Model Training

In [32]:
history = model.fit(
    x = [X_num_train, X_wcode_train, X_city_train],
    y = y_train_final,
    validation_data = ([X_num_val, X_wcode_val, X_city_val], y_val_final),
    epochs = 5,
    batch_size = 128,
    verbose=1
)

Epoch 1/5
657/657 ━━━━━━━━━━━━━━━━━━━━ 109s 166ms/step - loss: 0.0033 - mae: 0.0432 - val_loss: 0.0041 - val_mae: 0.0494
Epoch 2/5
657/657 ━━━━━━━━━━━━━━━━━━━━ 140s 164ms/step - loss: 0.0033 - mae: 0.0431 - val_loss: 0.0042 - val_mae: 0.0489
Epoch 3/5
657/657 ━━━━━━━━━━━━━━━━━━━━ 140s 161ms/step - loss: 0.0032 - mae: 0.0426 - val_loss: 0.0040 - val_mae: 0.0475
Epoch 4/5
657/657 ━━━━━━━━━━━━━━━━━━━━ 110s 167ms/step - loss: 0.0032 - mae: 0.0425 - val_loss: 0.0045 - val_mae: 0.0506
Epoch 5/5
657/657 ━━━━━━━━━━━━━━━━━━━━ 110s 167ms/step - loss: 0.0032 - mae: 0.0424 - val_loss: 0.0043 - val_mae: 0.0490


In [33]:
model.save("model.keras")

## 4. Evaluation

In [41]:
from sklearn.metrics import mean_absolute_error, r2_score

y_pred = model.predict([X_num_test, X_wcode_test, X_city_test], batch_size=16).ravel()
y_true = y_test_final.ravel()

mae = round(mean_absolute_error(y_true, y_pred), 4)
r2 = round(r2_score(y_true, y_pred), 2)

print(f"R2 Score:  {r2}")
print(f"Mean Squared Error: {mae}")

650/650 ━━━━━━━━━━━━━━━━━━━━ 21s 32ms/step
R2 Score:  0.71
Mean Squared Error: 0.0594
